In [ ]:
!pip install sentence-transformers chromadb groq pandas -q
print("Installation Completed")

Installation Completed


In [ ]:
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer
from groq import Groq
import os
print("All libraries imported successfully")

All libraries imported successfully


In [ ]:
from chromadb import api
GROQ_API_KEY="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"
os.environ["GROQ_API_KEY"]= GROQ_API_KEY
groq_client =Groq(api_key=GROQ_API_KEY)
print("Groq API client initialized.")
print("Note:If you see an authentication error later,double-check your API key.")

Groq API client initialized.
Note:If you see an authentication error later,double-check your API key.


In [ ]:
df = pd.read_csv('college_notes.csv')
print("Shape of Dataset:",df.shape)
print("\nColumn names:",df.columns.tolist())
print("\nFirst 3 rows:")
print(df.head(3))

Shape of Dataset: (15, 4)

Column names: ['note_id', 'subject', 'topic', 'content']

First 3 rows:
  note_id  ...                                            content
0    N001  ...  ETL stands for Extract Transform Load. It is t...
1    N002  ...  A database is an organized collection of data ...
2    N003  ...  Data cleaning involves fixing or removing inco...

[3 rows x 4 columns]


In [ ]:
print("Subjects in the Dataset:")
print(df['subject'].value_counts())
print("\nSample of Topics:")
print(df[['note_id', 'subject','topic']].to_string(index=False))
print("\nlength of the content (number of characters) for each note:")
print("\nFirst 3 rows:")
df['content_length'] = df['content'].apply(len)
print(df[['topic','content_length']].head(3).to_string(index=False))

Subjects in the Dataset:
subject
Data Engineering      5
Machine Learning      5
Generative AI         3
Python Programming    2
Name: count, dtype: int64

Sample of Topics:
note_id            subject                          topic
   N001   Data Engineering                  ETL Pipelines
   N002   Data Engineering                  SQL Databases
   N003   Data Engineering                  Data Cleaning
   N004   Data Engineering       APIs and Data Collection
   N005   Data Engineering           Big Data and PySpark
   N006   Machine Learning            Supervised Learning
   N007   Machine Learning               Model Evaluation
   N008   Machine Learning            Feature Engineering
   N009   Machine Learning                 Decision Trees
   N010   Machine Learning                  Random Forest
   N011      Generative AI          Large Language Models
   N012      Generative AI             Prompt Engineering
   N013      Generative AI Retrieval Augmented Generation
   N014 Python

In [ ]:
documents = df['content'].tolist()
ids = [f"note_{row['note_id']}" for row in df.to_dict('records')]
metadata = [
    {"subject":row['subject'],"topic":row['topic']} for row in df.to_dict('records')
]
print(f"Total chunks prepared: {len(documents)}")
print(f"First documnets ID : {ids[0]}")
print(f"First document metadata: {metadata[0]}")
print(f"First 100 chars of doc: {documents[0][:100]}...")

Total chunks prepared: 15
First documnets ID : note_N001
First document metadata: {'subject': 'Data Engineering', 'topic': 'ETL Pipelines'}
First 100 chars of doc: ETL stands for Extract Transform Load. It is the process of collecting raw data from different sourc...


In [ ]:
print("(Loading embedding model..)")
print("(This may take 30-60 seconds on first run-model is being downloaded)")
print("(Subsequent runs will be faster as the model is catched)")

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
print("Embedding model loaded successfully")
test_embedding = embedding_model.encode("This is a test sentence.")

print(f"Test embedding shape: {test_embedding.shape}")
print(f"Print fiirst 5 valuse of test embedding: {test_embedding[:5]}")

(Loading embedding model..)
(This may take 30-60 seconds on first run-model is being downloaded)
(Subsequent runs will be faster as the model is catched)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully
Test embedding shape: (384,)
Print fiirst 5 valuse of test embedding: [0.08429647 0.05795366 0.00449333 0.1058211  0.00708344]


In [ ]:
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(name="college_notes_rag")
print("ChromaDB client created.")
print(f"Collection name: college_notes_rag")
print(f"Documents in collection so far: {collection.count()}")

ChromaDB client created.
Collection name: college_notes_rag
Documents in collection so far: 0


In [ ]:
print("Generating embeddings for all 15 notes...")
print("This may take 15-30 seconds..")
embeddings_list = embedding_model.encode(documents, show_progress_bar=True)
print(f"\nEmbedding matrix shape: {embeddings_list.shape}")
embeddings_list = embeddings_list.tolist()

collection.add(
    documents=documents,
    ids=ids,
    metadatas=metadata,
    embeddings=embeddings_list
)
print(f"\nDocumnets successfully added to chromaDB.")
print(f"Total documents in collection:{collection.count()}")

Generating embeddings for all 15 notes...
This may take 15-30 seconds..


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Embedding matrix shape: (15, 384)

Documnets successfully added to chromaDB.
Total documents in collection:15


In [20]:
def retrieve_relevant_chunks(question,top_k=3):
    question_embedding=embedding_model.encode(question).tolist()
    results=collection.query(
        query_embeddings=question_embedding,
        n_results=top_k,
        include=['documents','metadatas']
    )
    return results

In [21]:
from sentence_transformers import SentenceTransformer
import chromadb

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(name="college_notes_rag")

test_question="What is ETL and how does it work in data engineering"
print(f'Test Quesstion:{test_question}')
results=retrieve_relevant_chunks(test_question,top_k=3)
print('\n Top 3 Retrieved Chunks ')
print("="*60)
for i,(doc,meta)in enumerate(zip(
    results['documents'][0],
    results['metadatas'][0]
)):

  print(f'\nResult {i+1}:')
  print(f'Subject:{meta["subject"]}')
  print(f'Topic:{meta["topic"]}')
  print(f'Content:{doc[:120]}...')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Test Quesstion:What is ETL and how does it work in data engineering

 Top 3 Retrieved Chunks 

Result 1:
Subject:Data Engineering
Topic:ETL Pipelines
Content:ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources transforming it i...

Result 2:
Subject:Data Engineering
Topic:APIs and Data Collection
Content:An API or Application Programming Interface allows two software applications to talk to each other. In data engineering ...

Result 3:
Subject:Python Programming
Topic:Data Visualization
Content:Data visualization is the process of representing data as charts graphs and visual formats. Python libraries like Matplo...


In [5]:
def generate_rag_answer(question, context):
  """
  Send the retrieved context and questions

  Parametrs:
      question (str): The user's question
      context (str): The retrieved context chunks (formatted string)


  Returns:
     answer (str) : The LLM's generated answer
  """

  system_prompt = """You are a helpful academic assistant for engineering students.

 You will be given context retrieved from a college knowledge base, and a student's question.

 RULES:
 1.Answer ONLY using the information provided in the context below.
 2.If the answer is not found in the context, say exactly:
 "I don't have enough information in my knowledge base to answer this question"
 3.Do not use your general training knowledge.
 4.Keep answers clear, accurate, and beginner-friendly.
 5.Mention which source the information came from which possible."""

  user_prompt = f"""Context: {context}

 Student's Question: {question}"""

  response = groq_client.chat.completions.create(
      model = "llama-3.1-8b-instant",
      messages = [
          {"role": "system", "content": system_prompt},
          {"role": "user", "content": user_prompt}
      ],
      temperature = 0.1,
      max_tokens = 500
  )
  answer=response.choices[0].message.content
  return answer
print("RAG generation function defined")

RAG generation function defined


In [6]:
def generate_rag_answer(question, context):
    system_prompt = """
You are a helpful academic assistant for engineering students.

You will be given context retrieved from a college knowledge base and a student's question.

RULES:
1. Answer only using the information provided in the context.
2. If the answer is not found in the context, say exactly:
   "I don't have enough information in my knowledge base to answer this question."
3. Do not use your general training knowledge.
4. Keep answers clear, accurate, and beginner-friendly.
5. Mention which source the answer came from.
"""

    user_prompt = f"""
Context:
{context}

Question:
{question}
"""

    response = client.chat.completions.create(
        model="llama-3.1-db-instant",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.1,
        max_tokens=400
    )

    return response.choices[0].message.content
print("done")

done


In [11]:
def ask_college_assistant(question, top_k=3, verbose=True):
    """
    Complete RAG pipeline: Given a question, retrieve relevant context and generate an answer

    Parameters:
        question (str): The user's question
        top_k (int): The number of context chunks to retrieve
        verbose (bool): If True, print

    Returns:
        answer (str): The LLM's generated answer
    """

    if verbose:
        print(f'Question: {question}')
        print("=" * 60)
        print("Step 1: Retrieving relevant documents...")